- Feature Engineering für die Modellierung pro Familiengruppe

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

In [6]:
# Pfade bei Bedarf anpassen
BASE_DIR = Path(r"C:\Users\nelid\Documents\Kaggle Competitions\Store Sales Competition\store-sales")
RAW_DIR       = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Rohdaten laden

train        = pd.read_csv(PROCESSED_DIR / "train_family_groups_holidays.csv",            parse_dates=["date"])
test         = pd.read_csv(PROCESSED_DIR / "test_family_groups_holidays.csv",             parse_dates=["date"])
stores       = pd.read_csv(RAW_DIR / "stores.csv")
oil          = pd.read_csv(RAW_DIR / "oil.csv",              parse_dates=["date"])
holidays     = pd.read_csv(RAW_DIR / "holidays_events.csv",  parse_dates=["date"])
transactions = pd.read_csv(RAW_DIR / "transactions.csv",     parse_dates=["date"])


C:\Users\nelid\AppData\Local\Temp\ipykernel_33240\3021666835.py:8: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  train        = pd.read_csv(PROCESSED_DIR / "train_family_groups_holidays.csv",            parse_dates=["date"])


In [7]:
#  Kalender- und Basis-Features ergänzen

def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["dayofweek"] = df["date"].dt.dayofweek
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
    df["month"] = df["date"].dt.month
    df["year"] = df["date"].dt.year
    df["dayofmonth"] = df["date"].dt.day
    df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)
    return df

train = add_calendar_features(train)
test  = add_calendar_features(test)


In [8]:
# Zeitfeatures wie im ersten Versuch

def add_time_features(df):
    df = df.copy()
    df["year"]       = df["date"].dt.year
    df["month"]      = df["date"].dt.month
    df["day"]        = df["date"].dt.day
    df["dayofweek"]  = df["date"].dt.dayofweek
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
    df["dayofyear"]  = df["date"].dt.dayofyear
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["quarter"]    = df["date"].dt.quarter

    # Zyklische Encodings
    df["month_sin"]  = np.sin(2 * np.pi * df["month"]     / 12)
    df["month_cos"]  = np.cos(2 * np.pi * df["month"]     / 12)
    df["dow_sin"]    = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * df["dayofweek"] / 7)
    df["woy_sin"]    = np.sin(2 * np.pi * df["weekofyear"]/ 52)
    df["woy_cos"]    = np.cos(2 * np.pi * df["weekofyear"]/ 52)
    df["doy_sin"]    = np.sin(2 * np.pi * df["dayofyear"] / 365)
    df["doy_cos"]    = np.cos(2 * np.pi * df["dayofyear"] / 365)
    return df

train = add_time_features(train)
test  = add_time_features(test)

# ─────────────────────────────────────────────────────────────────────────────
#    Payday-Features
#    Gehälter im öffentlichen Sektor: 15. und letzter Tag des Monats
# ─────────────────────────────────────────────────────────────────────────────
def add_payday_features(df):
    df = df.copy()
    day      = df["date"].dt.day
    last_day = (df["date"] + pd.offsets.MonthEnd(0)).dt.day

    df["is_payday_15th"]  = (day == 15).astype(int)
    df["is_payday_last"]  = (day == last_day).astype(int)
    df["is_payday"]       = ((df["is_payday_15th"] == 1) | (df["is_payday_last"] == 1)).astype(int)

    # Tage bis zum nächsten Zahltag
    days_until_15   = (15 - day).clip(lower=0)
    days_until_last = (last_day - day).clip(lower=0)
    df["days_until_payday"] = np.minimum(days_until_15, days_until_last)

    # Tage seit dem letzten Zahltag
    days_since_15 = (day - 15).clip(lower=0)
    # Vor dem 15.: Abstand zum letzten Tag des Vormonats
    df["days_since_payday"] = np.where(day >= 15, days_since_15, day - 1)
    # Sonderfall: nach letztem Tag → 0
    df["days_since_payday"] = np.where(
        df["is_payday_last"] == 1, 0, df["days_since_payday"]
    )

    # Kaufsog-Fenster: ±3 Tage um Zahltag
    df["is_payday_window"] = (
        (df["days_until_payday"] <= 3) | (df["days_since_payday"] <= 3)
    ).astype(int)

    return df

train = add_payday_features(train)
test  = add_payday_features(test)



In [9]:
# Sanity-Check
print("Zahltage im Train-Set:", train["is_payday"].sum() // train["store_nbr"].nunique() // train["family"].nunique())
print(train[["date","is_payday","days_since_payday","days_until_payday","is_payday_window"]]
      .drop_duplicates("date").query("is_payday==1").head(6))

Zahltage im Train-Set: 111
             date  is_payday  days_since_payday  days_until_payday  \
24948  2013-01-15          1                  0                  0   
53460  2013-01-31          1                  0                  0   
80190  2013-02-15          1                  0                  0   
103356 2013-02-28          1                  0                  0   
130086 2013-03-15          1                  0                  0   
158598 2013-03-31          1                  0                  0   

        is_payday_window  
24948                  1  
53460                  1  
80190                  1  
103356                 1  
130086                 1  
158598                 1  


In [10]:
# Ölpreis-Feature, interpoliert, kein Lookahead
oil = oil.rename(columns={"dcoilwtico": "oil_price"})
oil = (
    oil.set_index("date")
    .reindex(pd.date_range(oil["date"].min(), oil["date"].max(), freq="D"))
    .rename_axis("date")
    .reset_index()
)
oil["oil_price"] = oil["oil_price"].interpolate(method="linear")
# Öl-Lags (16, 21, 28 Tage) – kein Leakage im Test
for lag in [16, 21, 28]:
    oil[f"oil_lag_{lag}"] = oil["oil_price"].shift(lag)
# Rolling Mean des Ölpreises
oil["oil_rmean_28"] = oil["oil_price"].rolling(28, min_periods=1).mean()

train = train.merge(oil.drop(columns=["oil_price"]), on="date", how="left")
test  = test.merge(oil.drop(columns=["oil_price"]),  on="date", how="left")
# Wichtig: oil_price selbst weglassen (hätte Lookahead-Problem),
# stattdessen nur die Lags verwenden


In [11]:
# Transactions - Feature (nur Train – im Test unbekannt, daher Lags davon)

train = train.merge(transactions, on=["date", "store_nbr"], how="left")
test["transactions"] = np.nan  # Platzhalter, wird durch Lags ersetzt

In [12]:
# Speichere Basis-Datensatz vor Lags

train.to_csv(PROCESSED_DIR / "train_model_base_v3.csv", index=False)
test.to_csv( PROCESSED_DIR / "test_model_base_v3.csv",  index=False)
print("\nGespeichert: train_model_base_v3.csv, test_model_base_v3.csv")
print("Train shape:", train.shape)
print("Test shape: ", test.shape)


Gespeichert: train_model_base_v3.csv, test_model_base_v3.csv
Train shape: (3000888, 43)
Test shape:  (28512, 41)
